In [1]:
from google.colab import drive
drive.mount('/content/drive')
RAW_DIR = "/content/drive/MyDrive/yt-sentiment/raw"

Mounted at /content/drive


In [3]:
"""
YouTube comment ingestion script.
Pulls comments from a list of video IDs and saves:
  - raw/          one JSON file per video, untouched API responses (bronze layer)
  - comments.csv  flattened table of all comments (silver layer)

Setup:
1. Go to console.cloud.google.com, create a project (or reuse one).
2. Enable "YouTube Data API v3" under APIs & Services > Library.
3. Create an API key under APIs & Services > Credentials.
4. Set it as an environment variable:
     export YOUTUBE_API_KEY="your_key_here"
5. pip install requests

Quota note: commentThreads.list costs 1 unit per call. Default quota is
10,000 units/day, enough for roughly 1,000,000 comments at 100/page before
you'd hit the ceiling. Don't use search.list to find videos, it costs 100
units per call. Instead, manually collect video IDs by browsing YouTube
yourself and copying the part after ?v= in each URL.
"""

import os
import json
import csv
import time
import requests
from google.colab import userdata
API_KEY = userdata.get("YOUTUBE_API_KEY")
BASE_URL = "https://www.googleapis.com/youtube/v3/commentThreads"



In [4]:

# Fill this in manually. Search YouTube for "17 Agustus 2026" yourself,
# open each video, and copy the ID from the URL (the part after ?v=).
VIDEO_IDS = [
    "https://www.youtube.com/live/LlMJXBuyAPM?si=q2CkruKmzvMQklUO", # Kompas; 6.1 M views, 6 days ago
    "https://www.youtube.com/live/Hi_oCmIISJk?si=2Y-Ud_wTqFp83DsY", # KOMPASTV Madiun; 60K views, 6 days ago
    "https://www.youtube.com/live/0qGwRj_IORw?si=djpMi2pTvXyXslb3", # Sekretariat Presiden; 1.4 M views, 6 days ago
]

RAW_DIR = "raw"
os.makedirs(RAW_DIR, exist_ok=True)


In [6]:
import re
from urllib.parse import urlparse, parse_qs

def extract_video_id(url):
    """
    Extracts the YouTube video ID from various YouTube URL formats.
    Handles standard video URLs, shortened URLs, and live stream URLs.
    If the input is already a video ID, it returns it.
    """
    if len(url) == 11 and re.match(r'^[a-zA-Z0-9_-]+$', url):
        return url # Already a video ID

    parsed_url = urlparse(url)
    if parsed_url.hostname in ['www.youtube.com', 'youtube.com', 'm.youtube.com']:
        if parsed_url.path == '/watch':
            query = parse_qs(parsed_url.query)
            return query.get('v', [None])[0]
        elif parsed_url.path.startswith('/live/') or parsed_url.path.startswith('/v/') or parsed_url.path.startswith('/embed/') or parsed_url.path.startswith('/shorts/') :
            # Handles /live/VIDEO_ID, /v/VIDEO_ID, /embed/VIDEO_ID, /shorts/VIDEO_ID
            return parsed_url.path.split('/')[2] if len(parsed_url.path.split('/')) > 2 else None
    elif parsed_url.hostname == 'youtu.be':
        return parsed_url.path[1:] # Remove leading slash

    return None

def fetch_comments(video_id):
    """Fetch all comment threads for one video, paginating as needed."""
    all_items = []
    page_token = None

    while True:
        params = {
            "part": "snippet",
            "videoId": video_id,
            "key": API_KEY,
            "maxResults": 100,
            "order": "relevance",
            "textFormat": "plainText",
        }
        if page_token:
            params["pageToken"] = page_token

        resp = requests.get(BASE_URL, params=params)

        if resp.status_code == 403:
            print(f"  comments disabled or quota hit for {video_id}: {resp.text[:200]}")
            break
        resp.raise_for_status()

        data = resp.json()
        all_items.extend(data.get("items", []))

        # Save each raw page untouched. This is your bronze layer,
        # keep it separate from anything cleaned or transformed.
        with open(f"{RAW_DIR}/{video_id}_raw.json", "w", encoding="utf-8") as f:
            json.dump(all_items, f, ensure_ascii=False, indent=2)

        page_token = data.get("nextPageToken")
        if not page_token:
            break
        time.sleep(0.2)  # be polite, avoid rate limit errors

    return all_items


def flatten(video_id, raw_items):
    """Pull out the fields you actually need for the sentiment pipeline."""
    rows = []
    for item in raw_items:
        top = item["snippet"]["topLevelComment"]["snippet"]
        rows.append({
            "video_id": video_id,
            "comment_id": item["snippet"]["topLevelComment"]["id"],
            "author": top["authorDisplayName"],
            "text": top["textOriginal"],
            "published_at": top["publishedAt"],
            "like_count": top["likeCount"],
            "reply_count": item["snippet"]["totalReplyCount"],
        })
    return rows


def main():
    all_rows = []
    for vid_url in VIDEO_IDS:
        print(f"Fetching comments for {vid_url}...")
        actual_video_id = extract_video_id(vid_url)
        if not actual_video_id:
            print(f"  Skipping {vid_url}: could not extract valid video ID.")
            continue

        items = fetch_comments(actual_video_id)
        print(f"  got {len(items)} comment threads for video ID: {actual_video_id}")
        all_rows.extend(flatten(actual_video_id, items))

    with open("comments.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "video_id", "comment_id", "author", "text",
            "published_at", "like_count", "reply_count"
        ])
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"\nDone. {len(all_rows)} comments saved to comments.csv")
    print(f"Raw responses saved per video in {RAW_DIR}/")


if __name__ == "__main__":
    main()

Fetching comments for https://www.youtube.com/live/LlMJXBuyAPM?si=q2CkruKmzvMQklUO...
  got 187 comment threads for video ID: LlMJXBuyAPM
Fetching comments for https://www.youtube.com/live/Hi_oCmIISJk?si=2Y-Ud_wTqFp83DsY...
  got 4 comment threads for video ID: Hi_oCmIISJk
Fetching comments for https://www.youtube.com/live/0qGwRj_IORw?si=djpMi2pTvXyXslb3...
  got 612 comment threads for video ID: 0qGwRj_IORw

Done. 803 comments saved to comments.csv
Raw responses saved per video in raw/


In [9]:
from transformers import pipeline
import pandas as pd

sentiment = pipeline(
    "sentiment-analysis",
    model="w11wo/indonesian-roberta-base-sentiment-classifier",
    tokenizer="w11wo/indonesian-roberta-base-sentiment-classifier"
)

df = pd.read_csv("comments.csv")
results = sentiment(df["text"].astype(str).tolist(),
                    truncation=True, max_length=512, batch_size=16)

df["sentiment_label"] = [r["label"] for r in results]
df["sentiment_score"] = [r["score"] for r in results]
df.to_csv("comments_scored.csv", index=False)

print(df["sentiment_label"].value_counts())

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

sentiment_label
negative    304
positive    303
neutral     196
Name: count, dtype: int64


In [8]:
import matplotlib.pyplot as plt
import seaborn as sns

# Count the occurrences of each sentiment label
sentiment_counts = df['sentiment_label'].value_counts().reset_index()
sentiment_counts.columns = ['Sentiment', 'Count']

# Create the bar plot
plt.figure(figsize=(8, 6))
sns.barplot(x='Sentiment', y='Count', data=sentiment_counts, palette='viridis')
plt.title('Distribusi Sentimen Komentar YouTube')
plt.xlabel('Sentimen')
plt.ylabel('Jumlah Komentar')
plt.show()

KeyError: 'sentiment_label'

In [10]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("comments_scored.csv")

# Chart 1: overall distribution
counts = df["sentiment_label"].value_counts().reset_index()
counts.columns = ["sentiment", "count"]

fig1 = px.bar(
    counts, x="sentiment", y="count", color="sentiment",
    title="Distribusi Sentimen Komentar Upacara 17 Agustus 2026",
    color_discrete_map={"positive": "#2ecc71", "neutral": "#95a5a6", "negative": "#e74c3c"}
)
fig1.update_layout(showlegend=False)
fig1.show()

# Chart 2: breakdown per video, since you have 3
per_video = df.groupby(["video_id", "sentiment_label"]).size().reset_index(name="count")

fig2 = px.bar(
    per_video, x="video_id", y="count", color="sentiment_label",
    title="Sentimen per Video",
    color_discrete_map={"positive": "#2ecc71", "neutral": "#95a5a6", "negative": "#e74c3c"}
)
fig2.show()